# NarrativeNexus: Complete Text Analysis Pipeline

This notebook demonstrates the complete AI/ML/NLP pipeline used in the NarrativeNexus platform.

## Contents:
1. Text Preprocessing
2. Sentiment Analysis (VADER + TextBlob)
3. Topic Modeling (TF-IDF, LDA, NMF)
4. Text Summarization
5. Keyword Extraction
6. Visualizations

In [ ]:
# Import required libraries
import sys
sys.path.append('../backend/models')

import nltk
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from wordcloud import WordCloud

# Download NLTK data
nltk.download('punkt')
nltk.download('stopwords')
nltk.download('vader_lexicon')
nltk.download('wordnet')

# Import our custom modules
from preprocessing import preprocess_text, tokenize_sentences, get_word_frequency
from sentiment import analyze_sentiment, analyze_sentiment_by_sentence
from topic_model import extract_topics_tfidf, extract_topics_lda, extract_topics_nmf
from summarizer import summarize_text, get_summary_stats
from keywords import extract_keywords, extract_important_phrases

# Set visualization style
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

print("✅ All libraries imported successfully!")

## 1. Sample Text Data

Let's use a sample text about technology and AI:

In [ ]:
# Sample text for analysis
sample_text = """
Artificial intelligence and machine learning have revolutionized the technology industry in recent years. 
These powerful technologies are transforming how we live and work. Natural language processing enables 
computers to understand and generate human language with remarkable accuracy. Deep learning models have 
achieved breakthrough performance in image recognition, speech processing, and text analysis. Companies 
are increasingly adopting AI solutions to improve efficiency and create innovative products. However, 
there are also concerns about privacy, bias, and the ethical implications of AI systems. The future of 
AI holds both exciting opportunities and important challenges that society must address. Researchers are 
working on making AI more transparent, fair, and beneficial for everyone. Overall, artificial intelligence 
represents one of the most significant technological advances of our time.
"""

print("Original Text:")
print(sample_text)
print(f"\nLength: {len(sample_text.split())} words")
print(f"Sentences: {len(tokenize_sentences(sample_text))}")

## 2. Text Preprocessing

### Algorithm: Text Normalization and Cleaning
- Converts to lowercase
- Removes special characters and URLs
- Tokenizes words
- Removes stop words
- Applies lemmatization

In [ ]:
# Preprocess the text
cleaned_text = preprocess_text(sample_text)

print("Preprocessed Text:")
print(cleaned_text)
print(f"\nOriginal words: {len(sample_text.split())}")
print(f"After preprocessing: {len(cleaned_text.split())}")
print(f"Reduction: {len(sample_text.split()) - len(cleaned_text.split())} words removed")

## 3. Sentiment Analysis

### Algorithms Used:
1. **VADER** (Valence Aware Dictionary and sEntiment Reasoner)
   - Rule-based sentiment analysis
   - Optimized for social media text
   - Returns compound score from -1 (negative) to +1 (positive)

2. **TextBlob**
   - Pattern-based sentiment analysis
   - Returns polarity (-1 to +1) and subjectivity (0 to 1)

In [ ]:
# Analyze sentiment
sentiment_result = analyze_sentiment(sample_text)

print("=== SENTIMENT ANALYSIS RESULTS ===")
print(f"Overall Sentiment: {sentiment_result['label']}")
print(f"Compound Score: {sentiment_result['score']}")
print(f"Confidence: {sentiment_result['confidence']}")
print(f"\nDistribution:")
print(f"  Positive: {sentiment_result['positive']}%")
print(f"  Negative: {sentiment_result['negative']}%")
print(f"  Neutral: {sentiment_result['neutral']}%")
print(f"\nTextBlob Scores:")
print(f"  Polarity: {sentiment_result['textblob_polarity']}")
print(f"  Subjectivity: {sentiment_result['textblob_subjectivity']}")

In [ ]:
# Visualize sentiment distribution
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Pie chart
sentiments = ['Positive', 'Negative', 'Neutral']
values = [sentiment_result['positive'], sentiment_result['negative'], sentiment_result['neutral']]
colors = ['#48bb78', '#f56565', '#a0aec0']

axes[0].pie(values, labels=sentiments, autopct='%1.1f%%', colors=colors, startangle=90)
axes[0].set_title('Sentiment Distribution', fontsize=14, fontweight='bold')

# Bar chart
axes[1].bar(sentiments, values, color=colors, alpha=0.7)
axes[1].set_ylabel('Percentage (%)', fontsize=12)
axes[1].set_title('Sentiment Breakdown', fontsize=14, fontweight='bold')
axes[1].set_ylim(0, 100)

for i, v in enumerate(values):
    axes[1].text(i, v + 2, f'{v}%', ha='center', fontsize=11, fontweight='bold')

plt.tight_layout()
plt.show()

## 4. Topic Modeling

### Algorithms:

#### 4.1 TF-IDF (Term Frequency-Inverse Document Frequency)
**Formula:** `TF-IDF = TF × IDF`
- **TF (Term Frequency):** How often a term appears in a document
- **IDF (Inverse Document Frequency):** How unique the term is across all documents
- Higher TF-IDF = More important term

In [ ]:
# Extract topics using TF-IDF
tfidf_topics = extract_topics_tfidf(cleaned_text, n_topics=10)

print("=== TF-IDF TOPICS ===")
for i, topic in enumerate(tfidf_topics, 1):
    print(f"{i}. {topic['word']:15s} - Weight: {topic['weight']:.4f}")

#### 4.2 LDA (Latent Dirichlet Allocation)
**Probabilistic Model:**
- Assumes documents are mixtures of topics
- Topics are distributions over words
- Uses Bayesian inference to discover hidden topic structure

In [ ]:
# Extract topics using LDA
lda_topics = extract_topics_lda(cleaned_text, n_topics=5)

print("=== LDA TOPICS ===")
for topic in lda_topics:
    print(f"{topic['word']:15s} - Weight: {topic['weight']:.4f} (Topic {topic.get('topic_number', '')})}")

#### 4.3 NMF (Non-negative Matrix Factorization)
**Matrix Decomposition:**
- Decomposes document-term matrix into two matrices
- Document-topic matrix × Topic-term matrix
- All values must be non-negative

In [ ]:
# Extract topics using NMF
nmf_topics = extract_topics_nmf(cleaned_text, n_topics=5)

print("=== NMF TOPICS ===")
for topic in nmf_topics:
    print(f"{topic['word']:15s} - Weight: {topic['weight']:.4f} (Topic {topic.get('topic_number', '')})}")

In [ ]:
# Visualize topic weights (TF-IDF)
plt.figure(figsize=(12, 6))
words = [t['word'] for t in tfidf_topics[:10]]
weights = [t['weight'] for t in tfidf_topics[:10]]

plt.barh(words, weights, color='#667eea', alpha=0.7)
plt.xlabel('TF-IDF Weight', fontsize=12)
plt.title('Top 10 Topics by TF-IDF Weight', fontsize=14, fontweight='bold')
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

## 5. Text Summarization

### Algorithm: Extractive Summarization
**Steps:**
1. Split text into sentences
2. Calculate word frequency
3. Score each sentence based on word importance
4. Select top N% sentences
5. Return sentences in original order

In [ ]:
# Generate summary
summary = summarize_text(sample_text, ratio=0.3)

print("=== ORIGINAL TEXT ===")
print(sample_text)
print(f"\n{'='*50}\n")
print("=== SUMMARY ===")
print(summary)
print(f"\n{'='*50}\n")

# Get statistics
stats = get_summary_stats(sample_text, summary)
print("=== SUMMARY STATISTICS ===")
for key, value in stats.items():
    print(f"{key.replace('_', ' ').title()}: {value}")

In [ ]:
# Visualize compression
fig, ax = plt.subplots(figsize=(10, 6))

categories = ['Words', 'Sentences']
original = [stats['original_words'], stats['original_sentences']]
summary_vals = [stats['summary_words'], stats['summary_sentences']]

x = np.arange(len(categories))
width = 0.35

ax.bar(x - width/2, original, width, label='Original', color='#667eea', alpha=0.7)
ax.bar(x + width/2, summary_vals, width, label='Summary', color='#48bb78', alpha=0.7)

ax.set_ylabel('Count', fontsize=12)
ax.set_title(f'Original vs Summary\n(Compression: {stats["compression_ratio"]}%)', 
             fontsize=14, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(categories)
ax.legend()

plt.tight_layout()
plt.show()

## 6. Keyword Extraction

### Methods:
1. **Frequency-based:** Count word occurrences
2. **TF-IDF-based:** Statistical importance

In [ ]:
# Extract keywords
keywords = extract_keywords(cleaned_text, max_keywords=20, method='tfidf')

print("=== TOP KEYWORDS ===")
for i, kw in enumerate(keywords[:15], 1):
    print(f"{i:2d}. {kw['text']:20s} - Score: {kw['value']}")

In [ ]:
# Create word cloud
keyword_dict = {kw['text']: kw['value'] for kw in keywords}

wordcloud = WordCloud(width=800, height=400, background_color='white',
                     colormap='viridis', relative_scaling=0.5).generate_from_frequencies(keyword_dict)

plt.figure(figsize=(14, 7))
plt.imshow(wordcloud, interpolation='bilinear')
plt.axis('off')
plt.title('Keyword Cloud', fontsize=16, fontweight='bold', pad=20)
plt.tight_layout()
plt.show()

## 7. Complete Analysis Summary

Let's combine all results:

In [ ]:
# Create comprehensive analysis report
print("="*70)
print(" "*15 + "COMPLETE TEXT ANALYSIS REPORT")
print("="*70)

print("\n📊 DOCUMENT STATISTICS")
print("-" * 70)
print(f"Total Words: {len(sample_text.split())}")
print(f"Total Sentences: {len(tokenize_sentences(sample_text))}")
print(f"Unique Keywords: {len(keywords)}")

print("\n💭 SENTIMENT ANALYSIS")
print("-" * 70)
print(f"Overall Sentiment: {sentiment_result['label']}")
print(f"Confidence: {sentiment_result['confidence']}")
print(f"Compound Score: {sentiment_result['score']}")
print(f"Positive: {sentiment_result['positive']}% | "
      f"Negative: {sentiment_result['negative']}% | "
      f"Neutral: {sentiment_result['neutral']}%")

print("\n🎯 TOP 5 TOPICS (TF-IDF)")
print("-" * 70)
for i, topic in enumerate(tfidf_topics[:5], 1):
    print(f"{i}. {topic['word']} (Weight: {topic['weight']:.4f})")

print("\n📝 SUMMARY")
print("-" * 70)
print(summary)
print(f"\nCompression Ratio: {stats['compression_ratio']}%")

print("\n🔑 TOP 10 KEYWORDS")
print("-" * 70)
for i, kw in enumerate(keywords[:10], 1):
    print(f"{i:2d}. {kw['text']}")

print("\n" + "="*70)
print(" "*20 + "END OF REPORT")
print("="*70)

## Conclusion

This notebook demonstrates the complete NLP pipeline used in NarrativeNexus:

✅ **Text Preprocessing** - Cleaning and normalization  
✅ **Sentiment Analysis** - VADER + TextBlob algorithms  
✅ **Topic Modeling** - TF-IDF, LDA, and NMF  
✅ **Text Summarization** - Extractive summarization  
✅ **Keyword Extraction** - Statistical and frequency-based methods  
✅ **Visualizations** - Charts, graphs, and word clouds  

All these techniques work together to provide comprehensive text analysis and actionable insights!